# KZ Downloader — Colab Runner

Run yt-dlp downloads from any device (phone, Chromebook, someone else's PC) with **no Python install needed**.

**How to use:**
1. Generate a command on [kzdownloader.pages.dev](https://kzdownloader.pages.dev/) as usual.
2. Paste it into the cell below and click the ▶ play button.

That's it — one cell. It checks for yt-dlp/ffmpeg and installs them automatically the first time in a session (takes ~20s), then downloads. Every finished file pops up as its own browser download — nothing is saved to Google Drive.

**Works for:** public YouTube, Instagram, Facebook, Twitter/X, TikTok, and any generic yt-dlp-supported URL.
**Doesn't work for:** LinkedIn or private/login-gated content — that still needs the local Playwright scanner on a PC.

In [ ]:
#@title Paste your KZ Downloader command(s), then run this cell { display-mode: "form" }
commands = "" #@param {type:"string"}
#@markdown Paste one command per line, straight from the KZ Downloader website (any output-folder path in it is ignored and replaced automatically).
#@markdown
#@markdown Files download straight to your browser one by one, nothing is kept in Google Drive. Your browser may ask to allow multiple downloads the first time \u2014 click Allow.

import re
import os
import shutil
import subprocess
import sys
from google.colab import files as colab_files

SAVE_DIR = "/content/KZ Downloads"
os.makedirs(SAVE_DIR, exist_ok=True)

# \u2500\u2500 Auto-setup: install anything missing (skipped if already present) \u2500\u2500
def ensure_setup():
    need_ytdlp = shutil.which("yt-dlp") is None
    need_ffmpeg = shutil.which("ffmpeg") is None
    if need_ytdlp:
        print("\u2699\ufe0f Installing yt-dlp (first run only)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yt-dlp"], check=False)
    if need_ffmpeg:
        print("\u2699\ufe0f Installing ffmpeg (first run only)...")
        subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=False)
    if need_ytdlp or need_ffmpeg:
        print("\u2705 Setup complete.\n")

ensure_setup()

# \u2500\u2500 Run pasted command(s) \u2500\u2500
def fix_output_path(cmd: str, save_dir: str) -> str:
    # Strip any existing -o / --output flag and its value (quoted or not)
    cmd = re.sub(r'(-o|--output)\s+"(?:[^"\\\\]|\\\\.)*"', '', cmd)
    cmd = re.sub(r"(-o|--output)\s+'(?:[^'\\\\]|\\\\.)*'", '', cmd)
    cmd = re.sub(r'(-o|--output)\s+\S+', '', cmd)
    template = f'{save_dir}/%(title).150B [%(id)s].%(ext)s'
    return f'{cmd.strip()} -o "{template}"'

def list_files(root):
    found = set()
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            found.add(os.path.join(dirpath, fn))
    return found

lines = [l.strip() for l in commands.splitlines() if l.strip()]

if not lines:
    print("\u26a0\ufe0f Paste a command into the box above, then run this cell again.")
else:
    for i, raw_cmd in enumerate(lines, 1):
        cmd = raw_cmd
        if cmd.lower().startswith("yt-dlp"):
            cmd = cmd[len("yt-dlp"):].strip()
        cmd = fix_output_path(cmd, SAVE_DIR)
        full_cmd = f'yt-dlp {cmd}'
        print(f"\n{'='*60}\n[{i}/{len(lines)}] Running:\n{full_cmd}\n{'='*60}\n")

        before = list_files(SAVE_DIR)
        subprocess.run(full_cmd, shell=True)
        after = list_files(SAVE_DIR)
        new_files = sorted(after - before, key=lambda p: os.path.getmtime(p))

        for f in new_files:
            print(f"  \u2b07\ufe0f Downloading: {os.path.basename(f)}")
            colab_files.download(f)

    print("\n\u2705 Done.")

---
### Notes
- **Sharing with non-technical people:** send them this notebook's Colab link. They just need a Google account, paste the command, and click ▶.
- **Every device / new session** re-runs the install check automatically \u2014 it only actually installs anything the first time in that session, and just skips straight to downloading after that.
- **Session limits:** Colab disconnects after ~90 minutes idle and has a session cap of a few hours. Fine for batch downloads, not for anything long-running.
- **Nothing is saved to Google Drive** \u2014 files exist only in the temporary Colab session until they're downloaded to your browser, then they're gone from Colab.
- **Private/login-gated pages (LinkedIn, private IG/FB):** these still require the local Playwright scanner (`kz_scanner.py`) run on a PC with a real browser login \u2014 Colab has no way to log in interactively.
- **Batch downloads:** paste multiple commands (one per line) and they'll run one after another, each file downloading individually as it finishes.